# Decision Trees with LEGO - DEV A

This notebook contains the developer A part of the main notebook.

### Imports

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

import src.data as data
import src.plots as plots
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

### Constants

In [4]:
SEED = 42
N = 300
NOISE = 0.15

## Data Creation

In [7]:
clean_df = data.make_data(n=N, noise=0.0, seed=SEED)
noisy_df = data.make_data(n=N, noise=NOISE, seed=SEED)
display(clean_df.head())
noisy_df.head()

,stud_rows,stud_cols,colour,accepted,accepted_true,flipped
0,1,8,yellow,False,False,False
1,2,7,black,True,True,False
2,2,2,yellow,True,True,False
3,2,7,red,True,True,False
4,1,4,yellow,False,False,False


,stud_rows,stud_cols,colour,accepted,accepted_true,flipped
0,1,8,yellow,False,False,False
1,2,7,black,True,True,False
2,2,2,yellow,False,True,True
3,2,7,red,True,True,False
4,1,4,yellow,True,False,True


## Balance Check
With this, we can check if there is around 50% acceptance rate in the clean dataset. Meaning the rule is balanced
If it is, that means we can use accuracy as a metric to evaluate the model.

In [12]:
print(f"Acceptance rate: {clean_df['accepted_true'].mean()}")

Acceptance rate: 0.52


## Check if the Depth-3 Tree Expresses the Rule

In [ ]:
enc = data.make_encoder()
X = enc.fit_transform(clean_df[data.FEATURES])
names = enc.get_feature_names_out()
y = clean_df["accepted_true"]

tree = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(X, y)
print(export_text(tree, feature_names=list(names)))

used = {names[i] for i in tree.tree_.feature if i >= 0}   # negative ids are leaves
print("Features used:", used)
assert tree.score(X, y) == 1.0, "Depth 3 cannot express the rule"
assert {"stud_rows", "stud_cols"} <= used and any(u.startswith("colour_") for u in used)
print("Depth-3 tree recovers the rule with all three features")

|--- stud_rows <= 1.50
|   |--- stud_cols <= 4.50
|   |   |--- stud_cols <= 3.50
|   |   |   |--- class: False
|   |   |--- stud_cols >  3.50
|   |   |   |--- class: False
|   |--- stud_cols >  4.50
|   |   |--- colour_blue <= 0.50
|   |   |   |--- class: False
|   |   |--- colour_blue >  0.50
|   |   |   |--- class: False
|--- stud_rows >  1.50
|   |--- stud_cols <= 3.50
|   |   |--- colour_black <= 0.50
|   |   |   |--- class: True
|   |   |--- colour_black >  0.50
|   |   |   |--- class: True
|   |--- stud_cols >  3.50
|   |   |--- class: True

Features used: {'stud_rows', 'stud_cols', 'colour_black', 'colour_blue'}


AssertionError: Depth 3 cannot express the rule